# GutBrainIE – NER Ensemble (BERT token-classifier + GLiNER v2 span)
Obiettivo: **massimizzare i risultati finali** senza rifare training.

- BERT = *precision anchor*
- GLiNER v2 = *recall booster*

Pipeline:
1. Carica DEV
2. Predizioni BERT (dal tuo modello già allenato)
3. Predizioni GLiNER v2 (adapter LoRA già allenato)
4. Ensemble a livello di span + pruning finale
5. Valutazione con **`evaluate.py` ufficiale**


In [73]:
# 0) Imports
import os
import json
from pathlib import Path
from typing import Any, Dict, List
from collections import defaultdict
import re

## 1) Config

In [74]:
# =============================
# 1) CONFIG (EDIT ME)
# =============================
DATA_DIR = Path(r"C:/Users/super/Documents/UniPd/ATA/GutBrainIE/data/GutBrainIE_Full_Collection_2025/Annotations")
DEV_JSON = DATA_DIR / "Dev/json_format/dev.json"

# ---- BERT fine-tuned
BERT_MODEL_DIR = Path("models/bert_biomedbert_ner_label_weight")

# ---- GLiNER v2 fine-tuned (LoRA adapter directory)
GLINER_BASE_MODEL = "fastino/gliner2-base-v1"
GLINER_ADAPTER_DIR = Path(r"C:/Users/super/Documents/UniPd/ATA/GutBrainIE/src/ner/models/gliner_v2_finetuned/best")

# Output predictions
OUT_PRED_DIR = Path(r"C:/Users/super/Documents/UniPd/ATA/GutBrainIE/src/predictions")
OUT_PRED_DIR.mkdir(parents=True, exist_ok=True)

# ---- GLiNER inference threshold (tune for ensemble)
GLINER_THRESHOLD = 0.33
GLINER_INCLUDE_CONFIDENCE = True

# ---- Ensemble knobs
GLINER_MIN_ADD_SCORE = 1.01      # filtra aggiunte GLiNER troppo rumorose
ALLOW_EXPAND_SAME_LABEL = True   # se GLiNER contiene BERT (stessa label), espandi
EXPAND_MIN_SCORE = 0.60

APPLY_FINAL_PRUNE = True         # risolve overlap finali (BERT > GLiNER)

# evaluate.py (ufficiale). Metti qui il path nel tuo repo, se diverso.
EVALUATE_PY = Path("src/evaluate.py")
INCLUDE_CONFIDENCE = True


In [75]:
GLINER_REPLACE_MIN_SCORE = 0.70
GLINER_ADD_MIN_SCORE     = 0.85

BERT_REPLACE_MAX_SCORE   = 0.83  # sostituisco solo se BERT non è super sicuro


## 2) Labels (come nei tuoi notebook)

In [76]:
GLINER_LABELS = [
    "anatomical location",
    "animal",
    "bacteria",
    "biomedical technique",
    "chemical",
    "DDF",
    "dietary supplement",
    "drug",
    "food",
    "gene",
    "human",
    "microbiome",
    "statistical technique",
]


LABEL_MAPPING = {"disease, disorder or finding": "DDF"}

def normalize_label(label: str) -> str:
    return LABEL_MAPPING.get(label, label)

LEGAL_ENTITY_LABELS = {
    "anatomical location",
    "animal",
    "bacteria",
    "biomedical technique",
    "chemical",
    "DDF",
    "dietary supplement",
    "drug",
    "food",
    "gene",
    "human",
    "microbiome",
    "statistical technique",
}

## Load DEV

In [77]:
def load_json(path: Path) -> Dict[str, Any]:
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)

dev_data = load_json(DEV_JSON)
print("Loaded DEV docs:", len(dev_data))

Loaded DEV docs: 40


## BERT: Load model + tokenizer (inference only)

In [78]:
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModelForTokenClassification

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

bert_tokenizer = AutoTokenizer.from_pretrained(BERT_MODEL_DIR)
bert_model = AutoModelForTokenClassification.from_pretrained(BERT_MODEL_DIR).to(device)
bert_model.eval()

id2label = bert_model.config.id2label
label2id = bert_model.config.label2id
print("num labels:", len(id2label))


device: cuda


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 477.76it/s, Materializing param=classifier.weight]                                      


num labels: 27


### BERT: Two-pass thresholds (precision first, recall second)

In [79]:
LABEL_THRESH_HIGH = {
    "DDF": 0.88,
    "bacteria": 0.88,
    "statistical technique": 0.92,
    "biomedical technique": 0.82,
    "gene": 0.75,
    "food": 0.70,
    "chemical": 0.80,
    "dietary supplement": 0.85,
    "drug": 0.80,
    "microbiome": 0.78,
    "anatomical location": 0.78,
    "human": 0.70,
    "animal": 0.70,
}

LABEL_THRESH_RECALL = dict(LABEL_THRESH_HIGH)
LABEL_THRESH_RECALL.update({
    "DDF": 0.84,
    "chemical": 0.74,
    "biomedical technique": 0.76,
    "gene": 0.70,
    "food": 0.62,
})

DEFAULT_THRESH = 0.80

RECALL_LABELS = {"chemical", "biomedical technique", "gene", "food"}  # aggiungi "DDF" se serve

## BERT: Simple filters + label-specific postprocess

In [80]:
BAD_BACTERIA = {"bacteria", "micro", "microbes", "microorganisms", "genera", "taxa"}
BAD_CHEMICAL = {"metabolites", "neurotransmitters"}
BAD_DIETSUPP = {"nnss"}

def normalize_span(s: str) -> str:
    s = (s or "").strip().lower()
    s = re.sub(r"\s+", " ", s)
    return s

def apply_simple_filters(entities):
    cleaned = []
    for e in entities:
        s = normalize_span(e.get("text_span", ""))

        # drop HTML/markup garbage
        if "<" in s or ">" in s:
            continue

        # drop empty/very short spans
        if len(s) <= 1:
            continue

        # bacteria generic junk
        if e["label"] == "bacteria" and s in BAD_BACTERIA:
            continue

        # dietary supplement junk
        if e["label"] == "dietary supplement" and s in BAD_DIETSUPP:
            continue

        # chemical generic junk
        if e["label"] == "chemical" and s in BAD_CHEMICAL:
            continue

        cleaned.append(e)
    return cleaned


GENE_LIKE = re.compile(
    r"^(il-\d+|tnf-?α|ifn-?γ|tgf-?β\d*|snca|park7|dj-1|α-?synuclein|p-?α-?synuclein)$",
    re.IGNORECASE,
)

def postprocess_gene_vs_chemical(entities):
    for e in entities:
        if e["label"] == "chemical":
            s = normalize_span(e.get("text_span", ""))
            if GENE_LIKE.match(s):
                e["label"] = "gene"
    return entities

### BERT core: decode BIO + confidence score

In [81]:
## BERT: Core predictor (BIO decode + confidence score)
def predict_entities_with_scores(
        model,
        tokenizer,
        text: str,
        id2label: dict,
        label2id: dict,
        max_length: int = 512,
):
    """
    Output entity schema:
      start_idx (inclusive), end_idx (inclusive), label, text_span, score

    Score = mean token probability over the entity span.

    BIO repair:
      - I-X without active entity => start new entity as X
      - I-X with different active label => close current and start X
    """
    if not text:
        return []

    enc = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        return_offsets_mapping=True,
        max_length=max_length,
    )

    offsets = enc.pop("offset_mapping")[0].cpu().numpy()
    enc = {k: v.to(device) for k, v in enc.items()}

    with torch.no_grad():
        out = model(**enc)
        logits = out.logits[0]  # [T, C]
        probs = torch.softmax(logits, dim=-1)  # [T, C]
        pred_ids = torch.argmax(probs, dim=-1).cpu().numpy()
        probs_cpu = probs.cpu().numpy()

    labels = [id2label[int(i)] for i in pred_ids]

    entities = []
    current = None

    def _start_entity(ent_label: str, s: int, e: int, t_idx: int):
        prob_idx = label2id.get(f"B-{ent_label}", None)
        token_prob = float(probs_cpu[t_idx, prob_idx]) if prob_idx is not None else float(probs_cpu[t_idx].max())
        return {
            "start_idx": int(s),
            "end_idx": int(e) - 1,  # inclusive
            "label": ent_label,
            "text_span": text[int(s):int(e)],
            "_token_probs": [token_prob],
        }

    def _extend_entity(ent: dict, e: int, t_idx: int):
        ent_label = ent["label"]
        prob_idx = label2id.get(f"I-{ent_label}", None)
        token_prob = float(probs_cpu[t_idx, prob_idx]) if prob_idx is not None else float(probs_cpu[t_idx].max())
        ent["end_idx"] = int(e) - 1
        ent["text_span"] = text[ent["start_idx"]:int(e)]
        ent["_token_probs"].append(token_prob)

    for t_idx, (lab, (s, e)) in enumerate(zip(labels, offsets)):
        s = int(s);
        e = int(e)

        # special tokens
        if s == 0 and e == 0:
            continue
        if e <= s:
            continue

        if lab.startswith("B-"):
            if current is not None:
                entities.append(current)
            ent_label = lab[2:]
            current = _start_entity(ent_label, s, e, t_idx)

        elif lab.startswith("I-"):
            ent_label = lab[2:]

            if current is None:
                current = _start_entity(ent_label, s, e, t_idx)
                continue

            if ent_label != current["label"]:
                entities.append(current)
                current = _start_entity(ent_label, s, e, t_idx)
                continue

            _extend_entity(current, e, t_idx)

        else:
            if current is not None:
                entities.append(current)
                current = None

    if current is not None:
        entities.append(current)

    # add score
    for ent in entities:
        probs_list = ent.pop("_token_probs", [])
        ent["score"] = float(np.mean(probs_list)) if probs_list else 0.0

    return entities


### BERT: threshold filter + merge policy (two-pass)

In [82]:
## BERT: Threshold filter + merge policy
def filter_by_threshold_with_map(entities, label_thresh):
    out = []
    for e in entities:
        thr = label_thresh.get(e["label"], DEFAULT_THRESH)
        if float(e.get("score", 0.0)) >= float(thr):
            out.append(e)
    return out


def overlaps(a, b):
    # inclusive spans
    return not (a["end_idx"] < b["start_idx"] or b["end_idx"] < a["start_idx"])


def any_overlap(ent, kept):
    for k in kept:
        if k["location"] != ent["location"]:
            continue
        if overlaps(ent, k):
            return True
    return False


def merge_two_pass(ents_high, ents_rec, recall_labels):
    kept = list(ents_high)
    for e in ents_rec:
        if e["label"] not in recall_labels:
            continue
        if not any_overlap(e, kept):
            kept.append(e)
    return kept


#### BERT: Predict entities for a single segment (title/abstract) with two-pass

In [83]:
def predict_segment_entities_two_pass(model, tokenizer, text, location):
    ents_raw = predict_entities_with_scores(
        model=model,
        tokenizer=tokenizer,
        text=text,
        id2label=id2label,
        label2id=label2id,
        max_length=512,
    )

    # pass 1 (high precision)
    ents_high = filter_by_threshold_with_map(ents_raw, LABEL_THRESH_HIGH)
    ents_high = apply_simple_filters(ents_high)
    ents_high = postprocess_gene_vs_chemical(ents_high)
    for e in ents_high:
        e["location"] = location

    # pass 2 (recall)
    ents_rec = filter_by_threshold_with_map(ents_raw, LABEL_THRESH_RECALL)
    ents_rec = apply_simple_filters(ents_rec)
    ents_rec = postprocess_gene_vs_chemical(ents_rec)
    for e in ents_rec:
        e["location"] = location

    merged = merge_two_pass(ents_high, ents_rec, recall_labels=RECALL_LABELS)

    # ⚠️ NON togliere lo score qui (serve per l'ensemble).
    # Lo toglierai solo quando salvi la submission finale.
    return merged

### BERT: Predict DEV dataset (pmid -> {entities})

In [84]:
from tqdm import tqdm

def predict_dataset_bert_two_pass(dataset: Dict[str, Any]) -> Dict[str, Any]:
    preds = {}

    for pmid, article in tqdm(dataset.items(), desc="BERT two-pass inference"):
        title = (article.get("metadata", {}) or {}).get("title", "") or ""
        abstract = (article.get("metadata", {}) or {}).get("abstract", "") or ""

        ents = []
        if title.strip():
            ents += predict_segment_entities_two_pass(bert_model, bert_tokenizer, title, "title")
        if abstract.strip():
            ents += predict_segment_entities_two_pass(bert_model, bert_tokenizer, abstract, "abstract")

        # dedupe
        seen = set()
        dedup = []
        for e in ents:
            # dedupe key ignores text_span/score (score cambia spesso)
            k = (int(e["start_idx"]), int(e["end_idx"]), str(e["location"]), str(e["label"]))
            if k in seen:
                continue
            seen.add(k)
            dedup.append(e)

        preds[pmid] = {"entities": dedup}

    return preds

bert_predictions = predict_dataset_bert_two_pass(dev_data)
print("BERT docs predicted:", len(bert_predictions))
print("BERT total entities:", sum(len(v["entities"]) for v in bert_predictions.values()))

BERT two-pass inference: 100%|██████████| 40/40 [00:01<00:00, 23.72it/s]

BERT docs predicted: 40
BERT total entities: 980


SANITY CHECK

In [85]:
## GLiNER v2 inference (load base + LoRA adapter, predict dev)
from gliner2 import GLiNER2

txt = "Gut microbiota alterations are associated with Parkinson's disease."
base = GLiNER2.from_pretrained(GLINER_BASE_MODEL)
ft = GLiNER2.from_pretrained(GLINER_BASE_MODEL)
ft.load_adapter(str(GLINER_ADAPTER_DIR))

print(base.extract_entities(txt, GLINER_LABELS, threshold=0.33, include_spans=True))
print(ft.extract_entities(txt, GLINER_LABELS, threshold=0.33, include_spans=True))


You are using a model of type extractor to instantiate a model of type . This is not supported for all configurations of models and can yield errors.


🧠 Model Configuration
Encoder model      : microsoft/deberta-v3-base
Counting layer     : count_lstm_v2
Token pooling      : first


You are using a model of type extractor to instantiate a model of type . This is not supported for all configurations of models and can yield errors.


🧠 Model Configuration
Encoder model      : microsoft/deberta-v3-base
Counting layer     : count_lstm_v2
Token pooling      : first
{'entities': {'anatomical location': [], 'animal': [], 'bacteria': [], 'biomedical technique': [], 'chemical': [], 'DDF': [], 'dietary supplement': [], 'drug': [], 'food': [], 'gene': [], 'human': [{'text': "Parkinson's disease", 'start': 47, 'end': 66}], 'microbiome': [{'text': 'Gut microbiota', 'start': 0, 'end': 14}], 'statistical technique': []}}
{'entities': {'anatomical location': [], 'animal': [], 'bacteria': [{'text': 'Gut microbiota', 'start': 0, 'end': 14}], 'biomedical technique': [], 'chemical': [], 'DDF': [{'text': "Parkinson's disease", 'start': 47, 'end': 66}, {'text': 'Gut microbiota alterations', 'start': 0, 'end': 26}], 'dietary supplement': [], 'drug': [], 'food': [], 'gene': [], 'human': [], 'microbiome': [{'text': 'Gut microbiota', 'start': 0, 'end': 14}], 'statistical technique': []}}


##  GLiNER v2 inference

In [86]:
def dedupe_entities(ents: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    seen = set()
    out = []
    for e in ents:
        k = (int(e["start_idx"]), int(e["end_idx"]), str(e["location"]), str(e["label"]))
        if k in seen:
            continue
        seen.add(k)
        out.append(e)
    return out


def merge_adjacent_same_label(ents: List[Dict[str, Any]], title: str, abstract: str) -> List[Dict[str, Any]]:
    by_loc: Dict[str, List[Dict[str, Any]]] = {}
    for e in ents:
        by_loc.setdefault(e["location"], []).append(e)

    merged_all: List[Dict[str, Any]] = []
    for loc, group in by_loc.items():
        text = title if loc == "title" else abstract
        group = sorted(group, key=lambda x: (x["label"], x["start_idx"], x["end_idx"]))

        i = 0
        while i < len(group):
            cur = dict(group[i])
            i += 1

            while i < len(group) and group[i]["label"] == cur["label"]:
                nxt = group[i]

                # contiguous
                if nxt["start_idx"] == cur["end_idx"] + 1:
                    cur["end_idx"] = nxt["end_idx"]
                    cur["score"] = max(float(cur.get("score", 0.0)), float(nxt.get("score", 0.0)))
                    i += 1
                    continue

                # separated by one space
                if (
                        nxt["start_idx"] == cur["end_idx"] + 2
                        and text
                        and text[cur["end_idx"] + 1: cur["end_idx"] + 2] == " "
                ):
                    cur["end_idx"] = nxt["end_idx"]
                    cur["score"] = max(float(cur.get("score", 0.0)), float(nxt.get("score", 0.0)))
                    i += 1
                    continue

                break

            # recompute span from text if possible
            if text and 0 <= int(cur["start_idx"]) <= int(cur["end_idx"]) < len(text):
                cur["text_span"] = text[int(cur["start_idx"]): int(cur["end_idx"]) + 1]

            merged_all.append(cur)

    return merged_all


def postprocess_entities(ents: List[Dict[str, Any]], title: str, abstract: str) -> List[Dict[str, Any]]:
    # NB: volutamente NON facciamo prune overlap (ti dava soft recall migliore senza)
    ents = dedupe_entities(ents)
    ents = merge_adjacent_same_label(ents, title=title, abstract=abstract)
    ents = dedupe_entities(ents)
    return ents


def gliner2_extract_spans(
        extractor: GLiNER2,
        text: str,
        labels: List[str],
        threshold: float,
        location: str,
        include_confidence: bool = True,
) -> List[Dict[str, Any]]:
    if not text:
        return []

    result = extractor.extract_entities(
        text,
        labels,
        threshold=threshold,
        include_spans=True,
        include_confidence=include_confidence,
    )

    formatted: List[Dict[str, Any]] = []
    for raw_label, items in (result.get("entities", {}) or {}).items():
        norm_label = normalize_label(raw_label)
        if norm_label not in LEGAL_ENTITY_LABELS:
            continue

        for it in items:
            start = int(it["start"])
            end_exclusive = int(it["end"])
            end_inclusive = end_exclusive - 1  # GutBrainIE uses inclusive

            formatted.append(
                {
                    "start_idx": start,
                    "end_idx": end_inclusive,
                    "location": location,
                    "text_span": it.get("text", text[start:end_exclusive]),
                    "label": norm_label,
                    "score": float(it.get("confidence", 1.0)),
                }
            )

    return formatted


def predict_dataset_gliner2(
        extractor: GLiNER2,
        dataset: Dict[str, Any],
        labels: List[str],
        threshold: float,
        include_confidence: bool = True,
) -> Dict[str, Any]:
    preds: Dict[str, Any] = {}

    for pmid, article in tqdm(dataset.items(), desc=f"GLiNER2 inference (th={threshold})"):
        title = (article.get("metadata", {}) or {}).get("title", "") or ""
        abstract = (article.get("metadata", {}) or {}).get("abstract", "") or ""

        ents: List[Dict[str, Any]] = []
        ents += gliner2_extract_spans(extractor, title, labels, threshold, "title",
                                      include_confidence=include_confidence)
        ents += gliner2_extract_spans(extractor, abstract, labels, threshold, "abstract",
                                      include_confidence=include_confidence)

        ents = postprocess_entities(ents, title=title, abstract=abstract)

        preds[pmid] = {
            "entities": [
                {k: e[k] for k in ["start_idx", "end_idx", "location", "text_span", "label", "score"]}
                for e in ents
            ]
        }

    return preds


## Load GLiNER base + adapter
gliner_model = GLiNER2.from_pretrained(GLINER_BASE_MODEL)

if hasattr(gliner_model, "load_adapter"):
    gliner_model.load_adapter(str(GLINER_ADAPTER_DIR))
elif hasattr(gliner_model, "load_lora_adapter"):
    gliner_model.load_lora_adapter(str(GLINER_ADAPTER_DIR))
else:
    raise RuntimeError("No adapter loading method found (load_adapter/load_lora_adapter).")

print("Loaded GLiNER adapter:", GLINER_ADAPTER_DIR)

## Run GLiNER on DEV
gliner_predictions = predict_dataset_gliner2(
    extractor=gliner_model,
    dataset=dev_data,
    labels=GLINER_LABELS,
    threshold=GLINER_THRESHOLD,
    include_confidence=GLINER_INCLUDE_CONFIDENCE,
)

print("GLiNER docs predicted:", len(gliner_predictions))
print("GLiNER total entities:", sum(len(v["entities"]) for v in gliner_predictions.values()))


You are using a model of type extractor to instantiate a model of type . This is not supported for all configurations of models and can yield errors.


🧠 Model Configuration
Encoder model      : microsoft/deberta-v3-base
Counting layer     : count_lstm_v2
Token pooling      : first
Loaded GLiNER adapter: C:\Users\super\Documents\UniPd\ATA\GutBrainIE\src\ner\models\gliner_v2_finetuned\best


GLiNER2 inference (th=0.33): 100%|██████████| 40/40 [00:31<00:00,  1.25it/s]

GLiNER docs predicted: 40
GLiNER total entities: 1285


In [87]:
def ensemble_merge_replace_then_add(bert_preds, gliner_preds, dev_data):
    out = {}

    for pmid, article in dev_data.items():
        title = (article.get("metadata") or {}).get("title", "") or ""
        abstract = (article.get("metadata") or {}).get("abstract", "") or ""

        b_ents = list(bert_preds.get(pmid, {}).get("entities", []))
        g_ents = list(gliner_preds.get(pmid, {}).get("entities", []))

        # 1) REPLACE (same label + overlap)
        replaced = []
        for b in b_ents:
            best_g = None
            for g in g_ents:
                if g["location"] != b["location"]:
                    continue
                if g["label"] != b["label"]:
                    continue
                if not overlap(g, b):
                    continue
                if float(g.get("score", 0.0)) < GLINER_REPLACE_MIN_SCORE:
                    continue
                # sostituisco solo se BERT non è troppo sicuro
                if float(b.get("score", 1.0)) > BERT_REPLACE_MAX_SCORE:
                    continue

                # preferisci GLiNER se "aggiusta" i boundary:
                # - contiene BERT (espansione) oppure
                # - è contenuto (trim) oppure
                # - stessa lunghezza ma diversa posizione (spesso tokenization mismatch)
                if contains(g, b) or contains(b, g) or span_len(g) == span_len(b):
                    if (best_g is None) or float(g["score"]) > float(best_g["score"]):
                        best_g = g

            replaced.append(best_g if best_g is not None else b)

        # 2) ADD: solo GLiNER molto sicuro e solo se non overlap con nulla (dopo replace)
        final = list(replaced)
        # for g in g_ents:
        #     if float(g.get("score", 0.0)) < GLINER_ADD_MIN_SCORE:
        #         continue
        #     if any((e["location"] == g["location"] and overlap(e, g)) for e in final):
        #         continue
        #     final.append(g)

        # 3) ricostruisci text_span dai testi (come fai già)
        cleaned = []
        for e in final:
            loc = e["location"]
            text = title if loc == "title" else abstract
            s, t = int(e["start_idx"]), int(e["end_idx"])
            span = text[s:t+1] if text and 0 <= s <= t < len(text) else str(e.get("text_span", ""))
            cleaned.append({"start_idx": s, "end_idx": t, "location": loc, "text_span": span, "label": e["label"]})

        out[pmid] = {"entities": cleaned}

    return out
ensemble_predictions = ensemble_merge_replace_then_add(bert_predictions, gliner_predictions, dev_data)

## 6) Ensemble span-level
Regole (semplici ma efficaci):
- Parto da tutte le entità BERT
- Aggiungo entità GLiNER **non-overlap** con BERT (stessa `location`) e con `score >= GLINER_MIN_ADD_SCORE`
- Se overlap e *stessa label* e GLiNER **contiene** lo span BERT, posso espandere (opzionale)
- Alla fine faccio un pruning overlap: **BERT ha priorità**

In [88]:
def span_len(e):
    return int(e["end_idx"]) - int(e["start_idx"]) + 1

def overlap(a,b):
    return not (a["end_idx"] < b["start_idx"] or b["end_idx"] < a["start_idx"])

def contains(a,b):
    # a contains b
    return a["start_idx"] <= b["start_idx"] and a["end_idx"] >= b["end_idx"]


##  Save predictions

In [89]:
out_path = OUT_PRED_DIR / "ensemble_ner_2.json"
with out_path.open("w", encoding="utf-8") as f:
    json.dump(ensemble_predictions, f, ensure_ascii=False, indent=2)
print("Saved:", out_path)

Saved: C:\Users\super\Documents\UniPd\ATA\GutBrainIE\src\predictions\ensemble_ner_2.json


##  EVALUATION

In [90]:
import copy
import json
from pathlib import Path
from typing import Any, Dict, Tuple

# =========================
# OFFICIAL EVALUATOR (COPIED)
# =========================

LEGAL_ENTITY_LABELS = [
    "anatomical location",
    "animal",
    "bacteria",
    "biomedical technique",
    "chemical",
    "DDF",
    "dietary supplement",
    "drug",
    "food",
    "gene",
    "human",
    "microbiome",
    "statistical technique",
]

def remove_duplicated_entities(predictions: dict) -> None:
    removed_count = 0
    for pmid in list(predictions.keys()):
        seen = set()
        deduped = []
        for ent in predictions[pmid]["entities"]:
            # OFFICIAL: key WITHOUT label
            key = (ent["start_idx"], ent["end_idx"], ent["location"])
            if key not in seen:
                seen.add(key)
                deduped.append(ent)
            else:
                removed_count += 1
        predictions[pmid]["entities"] = deduped

    if removed_count > 0:
        print(f"=== Removed {removed_count} duplicated entities from predictions ===")

def remove_overlapping_entities(predictions: dict) -> None:
    removed_count = 0

    for pmid in list(predictions.keys()):
        original_len = len(predictions[pmid]["entities"])

        groups = {"title": [], "abstract": []}
        for ent in predictions[pmid]["entities"]:
            loc = ent["location"]
            groups[loc].append(ent)

        keepers = set()
        for loc in groups:
            group = sorted(groups[loc], key=lambda e: e["start_idx"])

            clusters = []
            cluster = []
            current_end = None

            for ent in group:
                if not cluster:
                    cluster = [ent]
                    current_end = ent["end_idx"]
                else:
                    # OFFICIAL: overlap if ent.start < current_end
                    if ent["start_idx"] < current_end:
                        cluster.append(ent)
                        if ent["end_idx"] > current_end:
                            current_end = ent["end_idx"]
                    else:
                        clusters.append(cluster)
                        cluster = [ent]
                        current_end = ent["end_idx"]

            if cluster:
                clusters.append(cluster)

            for clust in clusters:
                longest = clust[0]
                max_len = longest["end_idx"] - longest["start_idx"]
                for ent in clust[1:]:
                    length = ent["end_idx"] - ent["start_idx"]
                    if length > max_len:
                        longest = ent
                        max_len = length

                keepers.add((longest["start_idx"], longest["end_idx"], longest["location"]))

        deduped = []
        for ent in predictions[pmid]["entities"]:
            key = (ent["start_idx"], ent["end_idx"], ent["location"])
            if key in keepers:
                deduped.append(ent)
                keepers.remove(key)

        predictions[pmid]["entities"] = deduped
        removed_count += (original_len - len(deduped))

    if removed_count > 0:
        print(f"=== Removed {removed_count} overlapping entities ===")

def eval_submission_6_1_NER_from_predictions(
    predictions: Dict[str, Any],
    ground_truth: Dict[str, Any],
) -> Tuple[float, float, float, float, float, float]:
    """
    Same as official eval_submission_6_1_NER(path),
    but takes predictions dict directly (no file IO).
    """
    # IMPORTANT: the official functions mutate predictions -> deep copy
    predictions = copy.deepcopy(predictions)

    remove_duplicated_entities(predictions)
    remove_overlapping_entities(predictions)

    ground_truth_NER = {}
    count_annotated_entities_per_label = {}

    for pmid, article in ground_truth.items():
        ground_truth_NER.setdefault(pmid, [])
        for entity in article["entities"]:
            start_idx = int(entity["start_idx"])
            end_idx = int(entity["end_idx"])
            location = str(entity["location"])
            text_span = str(entity["text_span"])
            label = str(entity["label"])

            entry = (start_idx, end_idx, location, text_span, label)
            ground_truth_NER[pmid].append(entry)

            count_annotated_entities_per_label[label] = count_annotated_entities_per_label.get(label, 0) + 1

    count_predicted_entities_per_label = {lab: 0 for lab in count_annotated_entities_per_label.keys()}
    count_true_positives_per_label = {lab: 0 for lab in count_annotated_entities_per_label.keys()}

    for pmid in predictions.keys():
        entities = predictions[pmid]["entities"]
        for entity in entities:
            start_idx = int(entity["start_idx"])
            end_idx = int(entity["end_idx"])
            location = str(entity["location"])
            text_span = str(entity["text_span"])
            label = str(entity["label"])

            if label not in LEGAL_ENTITY_LABELS:
                raise NameError(f'{pmid} - Illegal label {label} for entity: {entity}')

            if label in count_predicted_entities_per_label:
                count_predicted_entities_per_label[label] += 1

            entry = (start_idx, end_idx, location, text_span, label)
            if entry in ground_truth_NER[pmid]:
                count_true_positives_per_label[label] += 1

    count_annotated_entities = sum(count_annotated_entities_per_label.values())
    count_predicted_entities = sum(count_predicted_entities_per_label.values())
    count_true_positives = sum(count_true_positives_per_label.values())

    micro_precision = count_true_positives / (count_predicted_entities + 1e-10)
    micro_recall = count_true_positives / (count_annotated_entities + 1e-10)
    micro_f1 = 2 * ((micro_precision * micro_recall) / (micro_precision + micro_recall + 1e-10))

    precision = recall = f1 = 0.0
    n = 0
    for label in count_annotated_entities_per_label.keys():
        n += 1
        current_precision = count_true_positives_per_label[label] / (count_predicted_entities_per_label[label] + 1e-10)
        current_recall = count_true_positives_per_label[label] / (count_annotated_entities_per_label[label] + 1e-10)
        precision += current_precision
        recall += current_recall
        f1 += 2 * ((current_precision * current_recall) / (current_precision + current_recall + 1e-10))

    precision /= n
    recall /= n
    f1 /= n

    return precision, recall, f1, micro_precision, micro_recall, micro_f1


def official_metrics_dict(
    predictions: Dict[str, Any],
    ground_truth: Dict[str, Any],
) -> Dict[str, float]:
    p, r, f1, mp, mr, mf1 = eval_submission_6_1_NER_from_predictions(predictions, ground_truth)
    return {
        "macro_precision": float(p),
        "macro_recall": float(r),
        "macro_f1": float(f1),
        "micro_precision": float(mp),
        "micro_recall": float(mr),
        "micro_f1": float(mf1),
    }


In [91]:
print("BERT-only:", official_metrics_dict(bert_predictions, dev_data))
print("GLiNER-only:", official_metrics_dict(gliner_predictions, dev_data))
print("ENSEMBLE:", official_metrics_dict(ensemble_predictions, dev_data))


BERT-only: {'macro_precision': 0.8678949614516107, 'macro_recall': 0.7136848455066089, 'macro_f1': 0.7676700326451387, 'micro_precision': 0.8867346938774605, 'micro_recall': 0.7779767233660897, 'micro_f1': 0.8288030519291519}
=== Removed 131 duplicated entities from predictions ===
=== Removed 48 overlapping entities ===
GLiNER-only: {'macro_precision': 0.44927258315359, 'macro_recall': 0.5150855776716813, 'macro_f1': 0.4457920913546227, 'micro_precision': 0.5298372513561908, 'micro_recall': 0.5246195165621732, 'micro_f1': 0.5272154745338493}
ENSEMBLE: {'macro_precision': 0.8708535413332561, 'macro_recall': 0.7158215976433552, 'macro_f1': 0.7701514222232944, 'micro_precision': 0.8877551020407257, 'micro_recall': 0.7788719785138067, 'micro_f1': 0.8297567953721656}


## analysis

In [92]:
from collections import Counter, defaultdict

def gold_entries(ground_truth):
    gold = defaultdict(set)
    for pmid, art in ground_truth.items():
        for e in art["entities"]:
            key = (int(e["start_idx"]), int(e["end_idx"]), e["location"], e["text_span"], e["label"])
            gold[pmid].add(key)
    return gold

def pred_entries(preds):
    pred = defaultdict(list)
    for pmid, obj in preds.items():
        for e in obj["entities"]:
            key = (int(e["start_idx"]), int(e["end_idx"]), e["location"], e["text_span"], e["label"])
            pred[pmid].append((key, e))
    return pred

def analyze_errors(preds, ground_truth):
    gold = gold_entries(ground_truth)
    pred = pred_entries(preds)

    by_label = defaultdict(lambda: Counter(tp=0, fp=0, fn=0))
    by_src   = defaultdict(lambda: Counter(tp=0, fp=0))
    fp_examples = defaultdict(list)  # label -> list of (pmid, loc, span, src)

    # TP/FP
    for pmid, items in pred.items():
        for key, e in items:
            lab = key[4]
            src = e.get("_src", "unknown")
            if key in gold.get(pmid, set()):
                by_label[lab]["tp"] += 1
                by_src[src]["tp"] += 1
            else:
                by_label[lab]["fp"] += 1
                by_src[src]["fp"] += 1
                if len(fp_examples[lab]) < 30:
                    fp_examples[lab].append((pmid, key[0], key[1], key[2], key[3], lab, src))



    # FN
    for pmid, gset in gold.items():
        pkeys = set(k for k,_ in pred.get(pmid, []))
        for g in gset:
            lab = g[4]
            if g not in pkeys:
                by_label[lab]["fn"] += 1

    # calcola precision/recall per label
    per_label_stats = {}
    for lab, c in by_label.items():
        tp, fp, fn = c["tp"], c["fp"], c["fn"]
        p = tp / (tp + fp + 1e-10)
        r = tp / (tp + fn + 1e-10)
        f1 = 2*p*r/(p+r+1e-10)
        per_label_stats[lab] = dict(tp=tp, fp=fp, fn=fn, precision=p, recall=r, f1=f1)

    return per_label_stats, by_src, fp_examples


In [98]:
def ensemble_merge_replace_then_add_debug(bert_preds, gliner_preds, dev_data):
    out = {}

    for pmid, article in dev_data.items():
        title = (article.get("metadata") or {}).get("title", "") or ""
        abstract = (article.get("metadata") or {}).get("abstract", "") or ""

        b_ents = list(bert_preds.get(pmid, {}).get("entities", []))
        g_ents = list(gliner_preds.get(pmid, {}).get("entities", []))

        replaced = []
        for b in b_ents:
            best_g = None
            for g in g_ents:
                if g["location"] != b["location"]:
                    continue
                if g["label"] != b["label"]:
                    continue
                if not overlap(g, b):
                    continue
                if float(g.get("score", 0.0)) < GLINER_REPLACE_MIN_SCORE:
                    continue
                if float(b.get("score", 1.0)) > BERT_REPLACE_MAX_SCORE:
                    continue
                if not (contains(g, b) or contains(b, g)):
                    continue
                if b["label"] == "statistical technique" and float(b.get("score", 1.0)) > 0.75:
                     continue
                if best_g is None or float(g["score"]) > float(best_g["score"]):
                    best_g = g

            if best_g is not None:
                g2 = dict(best_g)
                g2["_src"] = "gliner_replace"
                replaced.append(g2)
            else:
                b2 = dict(b)
                b2["_src"] = "bert"
                replaced.append(b2)

        final = list(replaced)


        cleaned = []
        for e in final:
            loc = e["location"]
            text = title if loc == "title" else abstract
            s, t = int(e["start_idx"]), int(e["end_idx"])
            span = text[s:t+1] if text and 0 <= s <= t < len(text) else str(e.get("text_span", ""))
            cleaned.append({
                "start_idx": s,
                "end_idx": t,
                "location": loc,
                "text_span": span,
                "label": e["label"],
                "_src": e["_src"],          # <-- chiave debug
            })

        out[pmid] = {"entities": cleaned}

    return out


In [99]:
ensemble_predictions_debug = ensemble_merge_replace_then_add_debug(
    bert_predictions,
    gliner_predictions,
    dev_data
)


In [100]:
stats, src_stats, fp_examples = analyze_errors(ensemble_predictions_debug, dev_data)

# Ordina label per peggio precision (o per fp più alti)
worst = sorted(stats.items(), key=lambda x: x[1]["precision"])
for lab, s in worst:
    print(lab, s)

print("\nBy source:", src_stats)

print("\nFP examples for 'chemical':")
for ex in fp_examples["chemical"][:10]:
    print(ex)


gene {'tp': 22, 'fp': 13, 'fn': 17, 'precision': 0.6285714285696327, 'recall': 0.5641025641011177, 'f1': 0.5945945945431337}
chemical {'tp': 75, 'fp': 28, 'fn': 56, 'precision': 0.7281553398051183, 'recall': 0.5725190839690286, 'f1': 0.6410256409758092}
biomedical technique {'tp': 21, 'fp': 5, 'fn': 15, 'precision': 0.8076923076892012, 'recall': 0.5833333333317129, 'f1': 0.6774193547878251}
bacteria {'tp': 43, 'fp': 9, 'fn': 11, 'precision': 0.8269230769214867, 'recall': 0.7962962962948217, 'f1': 0.8113207546654683}
anatomical location {'tp': 68, 'fp': 11, 'fn': 8, 'precision': 0.8607594936697965, 'recall': 0.8947368421040859, 'f1': 0.8774193547875963}
drug {'tp': 54, 'fp': 8, 'fn': 6, 'precision': 0.8709677419340791, 'recall': 0.8999999999985, 'f1': 0.8852459015879064}
dietary supplement {'tp': 16, 'fp': 2, 'fn': 11, 'precision': 0.8888888888839507, 'recall': 0.5925925925903979, 'f1': 0.7111111110599508}
microbiome {'tp': 119, 'fp': 10, 'fn': 8, 'precision': 0.9224806201543237, 'recal

In [101]:
for lab in ["chemical", "biomedical technique", "statistical technique"]:
    print("\nFP examples for", lab)
    for ex in fp_examples[lab][:10]:
        print(ex)



FP examples for chemical
('37212075', 238, 250, 'abstract', 'secretory IgA', 'chemical', 'bert')
('37212075', 253, 256, 'abstract', 'SIgA', 'chemical', 'bert')
('36978911', 943, 948, 'abstract', 'intact', 'chemical', 'bert')
('36978911', 950, 958, 'abstract', 'flavonoid', 'chemical', 'bert')
('31610228', 552, 556, 'abstract', 'MCP-1', 'chemical', 'bert')
('31610228', 559, 564, 'abstract', 'IL-1RA', 'chemical', 'bert')
('31610228', 567, 571, 'abstract', 'IL-1β', 'chemical', 'bert')
('31610228', 1087, 1091, 'abstract', 'MCP-1', 'chemical', 'bert')
('26923630', 1150, 1177, 'abstract', 'bacteria-derived metabolites', 'chemical', 'bert')
('26923630', 1211, 1223, 'abstract', 'lipid species', 'chemical', 'bert')

FP examples for biomedical technique
('37212075', 677, 689, 'abstract', 'amplification', 'biomedical technique', 'bert')
('37577447', 339, 365, 'abstract', '16S rRNA sequence libraries', 'biomedical technique', 'bert')
('31610228', 609, 614, 'abstract', 'custom', 'biomedical techniq